# Parsing MISO Emergency Historical Information

### Imports and Notebook Settings


In [4]:
%load_ext autoreload
%autoreload 2

In [5]:
import os 
import re
from pathlib import Path

In [6]:
import pandas as pd
import numpy as np 

import matplotlib.pyplot as plt
# import seaborn as sns 
import plotly.express as px
import pprint 

In [7]:
ROOT = Path.cwd().parent

In [8]:
style_path = ( ROOT 
              / 'notebooks' 
              / 'styler.mplstyle'
              )
plt.style.use(style_path)

### Load PDF

In [11]:
DATA_PATH = ROOT / 'data' / 'samples' / 'Capacity_Emergency_Historical_Information.pdf'

In [62]:
from pypdf import PdfReader

reader = PdfReader(DATA_PATH)
pages = []
for page in reader.pages:
    text = page.extract_text() or ""

    lines = text.splitlines()

    print(lines)
    # remove first few header lines
    lines = lines[9:]

    pages.append("\n".join(lines))


full_text = "\n".join(pages)

[' ', 'Maximum Generation Emergency Declarations ', 'through June 2024 ', '(Updated 08/30/2024) ', ' ', '1 ', 'No Maximum Generation Emergency Declarations issued in 2015 ', 'SO-P-EOP-00-002 ', 'Contents ', '2009 .............................................................................................. 4 ', '2010 .............................................................................................. 5 ', '2011 .............................................................................................. 6 ', '2012 .............................................................................................. 7 ', '2013 .............................................................................................. 8 ', '2014 .............................................................................................. 9 ', '2016 ............................................................................................ 10 ', '2017 ..............................................

In [63]:
print(full_text)

2009 .............................................................................................. 4 
2010 .............................................................................................. 5 
2011 .............................................................................................. 6 
2012 .............................................................................................. 7 
2013 .............................................................................................. 8 
2014 .............................................................................................. 9 
2016 ............................................................................................ 10 
2017 ............................................................................................ 11 
2018 ............................................................................................ 13 
2019 .......................................................................

In [83]:
def parse_events(text: str) -> list[str]:
    bullets = re.split(r"\n\s*[•▪◦]\s*", text)

    date_start_pattern = re.compile(
        r"^\s*\d{2}/\d{2}/\d{4}\s+"
        r"\d{1,2}:\d{2}\s*(?:EST)?\s*[-–—]\s*"
        r"\d{1,2}:\d{2}\s+EST"
    )
    events = []
    current_event = None

    for bullet in bullets:
        bullet = bullet.strip()

        if not bullet:
            continue

        if date_start_pattern.match(bullet):
            if current_event is not None:
                events.append(current_event.strip())

            current_event = bullet

        elif current_event is not None:
            current_event += "\n" + bullet

    if current_event is not None:
        events.append(current_event.strip())

    return events

In [84]:
events = parse_events(full_text)

In [87]:
for i in events[:25]:
    print('=' * 20)
    print(i)

01/13/2009 18:00 – 20:30 EST – West Region Maximum Generation Emergency 
ALERT The reason for the Alert is cold temperatures and generation loss.
08/11/2010 14:00 – 20:00 EST – Sub-Area Maximum Generation Emergency 
ALERT Declared for the subarea of FE-Northeast Ohio due to generation loss and 
forced outages.
06/08/2011 12:00-19:00 EST –MISO RC declared a Maximum Generation 
Emergency Alert for the following entities: Central Region Market area(s) of: AMIL, 
AMMO, AMRN, BREC, CIN, CWLD, CWLP, HE, IPL, SIGE, SIPC and East Region 
Market area(s) of: ALTE, MECS, MGE, NIPS, UPPC, WEC, WPS and West Region 
Market area(s) of: ALTW, DPC, MEC, MPW due to Above Normal Temps.
This Alert does not include the following LBA's: GRE, NSP, SMP, MP, OTP, 
MDU, which are all in the Minnesota and the Dakotas area. Temperatures 
have dropped considerably from yesterday in this area.
07/18/2011 14:00-15:00 EST - Market Footprint Maximum Generation Emergency 
ALERT The reason for the Alert is because of Ab

In [74]:
print(len(events))

115
